# K=4 study: overselection control and power on a four-level class ladder
Generates results/ksweep_k4/. Independent row splits, d=4, five DGPs of orders 1-4.

In [ ]:
import numpy as np, json, csv, time, hashlib, zlib, inspect, sys, platform
import scipy
from itertools import product as iproduct
from scipy import stats
import os
OUTBASE = '/content/drive/MyDrive/ORDER_SWEEP/results'

def monomial_exps(d, D, k):
    return [c for c in iproduct(range(D+1), repeat=d)
            if sum(c) <= D and sum(1 for x in c if x > 0) <= k]
def poly_design(X, D, k):
    exps = monomial_exps(X.shape[1], D, k); cols = []
    for e in exps:
        col = np.ones(X.shape[0])
        for j, p in enumerate(e):
            if p > 0: col = col * X[:, j]**p
        cols.append(col)
    return np.column_stack(cols), exps.index(tuple([0]*X.shape[1]))
def select_order(X, h, K, S=10, alpha=0.05, D=4, train_frac=0.75):
    n = X.shape[0]; n_tr = int(train_frac*n); n_te = n - n_tr
    designs = {k: poly_design(X, D, k) for k in range(1, K+1)}; r2 = {}
    for s in range(S):
        idx = np.random.default_rng(s).permutation(n); tr, te = idx[:n_tr], idx[n_tr:]; out = {}
        for k in range(1, K+1):
            P0, ci = designs[k]; mu = P0[tr].mean(0); sd = P0[tr].std(0); sd[sd == 0] = 1.0
            P = (P0 - mu)/sd; P[:, ci] = 1.0; hm = h[tr].mean()
            beta, *_ = np.linalg.lstsq(P[tr], h[tr]-hm, rcond=None); resid = (h[te]-hm) - P[te]@beta
            out[k] = 1.0 - float((resid@resid)/np.sum((h[te]-h[te].mean())**2))
        r2[s] = out
    corr = 1.0/S + n_te/n_tr; pf = {k: designs[k][0].shape[1] for k in range(1, K+1)}
    om = float(np.mean([1.0 - r2[s][K] for s in range(S)])); stat = {}
    for k in range(1, K):
        g = np.array([r2[s][K] - r2[s][k] for s in range(S)]); m = g.mean(); v = g.var(ddof=1)*corr
        opt = (pf[K] - pf[k])*om/n_tr
        if v > 0:
            t = m/np.sqrt(v); p = 1.0 - stats.t.cdf(t, df=S-1); ub = m + stats.t.ppf(1-alpha, df=S-1)*np.sqrt(v) + opt
        else:
            p = 0.0 if m > 0 else 1.0; ub = m + opt
        stat[k] = {"mean": float(m), "p": float(p), "ub": float(ub)}
    khat, ubc = K, None
    for k in range(1, K):
        if stat[k]["p"] > alpha: khat, ubc = k, stat[k]["ub"]; break
    return khat, stat, ubc

# ---- K=4 study ----
OUT = os.path.join(OUTBASE, "ksweep_k4"); os.makedirs(OUT, exist_ok=True)
def make_X(n, dep, rng):
    Z = rng.standard_normal((n, 4))
    if dep.startswith("pair"):
        rho = float(dep[4:]); Z[:, 2] = rho*Z[:, 0] + np.sqrt(1-rho**2)*Z[:, 2]
    return Z
def make_h(X, dgp, sig, rng):
    x1, x2, x3, x4 = X.T
    f = {"o1": x1+np.tanh(x2)-0.5*x3+0.3*x4, "o2": x1*x2+np.tanh(x3)+0.5*x4,
         "o3": x1*x2*x3+x4, "o4": x1*x2*x3*x4, "o4mix": x1*x2*x3*x4+x1+x2*x3}[dgp]
    return f + sig*rng.standard_normal(len(f))
TRUE={"o1":1,"o2":2,"o3":3,"o4":4,"o4mix":4}
N,R=20_000,20; DGPS=["o1","o2","o3","o4","o4mix"]; DEPS=["indep","pair0.5"]; SIGS=[0.0,0.5]
rows=[]
for dgp in DGPS:
  for dep in DEPS:
    for sig in SIGS:
      for rep in range(R):
        seed=(40_000+zlib.crc32(f"k4|{dgp}|{dep}|{sig}".encode())+rep*977)%2**32
        rng=np.random.default_rng(seed); X=make_X(N,dep,rng); h=make_h(X,dgp,sig,rng)
        kh,st,ub=select_order(X,h,K=4)
        rows.append({"experiment":"ksweep_k4","dgp":dgp,"dependence":dep,"sigma":sig,"rep":rep,
                     "true_order":TRUE[dgp],"khat":kh,"rem1_p":st[1]["p"],"rem2_p":st[2]["p"],
                     "rem3_p":st[3]["p"],"ub_cert":"" if ub is None else ub})
with open(os.path.join(OUT,"per_seed_k4.csv"),"w",newline="") as f:
    w=csv.DictWriter(f,fieldnames=list(rows[0].keys())); w.writeheader(); w.writerows(rows)
print("wrote", len(rows), "rows to ksweep_k4/per_seed_k4.csv")
